# Día 7 · Unidad 7 — pandas y NumPy (II): Series

**Objetivos de aprendizaje**
- Explicar qué es una `Series` de pandas y cómo se relaciona con los arrays de NumPy que acabas de ver.
- Crear Series desde listas, diccionarios y archivos, y leer sus atributos básicos.
- Usar los métodos más habituales (`head`/`tail`/`sort_values`/`sort_index`/`apply`).
- Aplicar operaciones vectorizadas y filtros booleanos sobre una Series, reconociendo el paralelismo con NumPy.

**Duración estimada:** 60 minutos.

**Requisitos previos:** `01_teoria_numpy.ipynb` de este mismo día.

## 1. Qué es una Series

Una [`Series`](https://pandas.pydata.org/docs/reference/series.html) es una estructura unidimensional y **etiquetada**, capaz de contener cualquier tipo de dato (números, texto, fechas...). Es, en esencia, un array de NumPy con un índice explícito pegado a cada valor.

> **Principio básico de pandas**: la alineación entre datos e índice es intrínseca a las Series — el vínculo entre un valor y su etiqueta de índice no se rompe salvo que tú lo rompas explícitamente.

Empezamos importando lo necesario. En pandas casi siempre verás `numpy` importado también, por lo que viste en el notebook anterior: pandas se apoya en NumPy por dentro.

In [1]:
import numpy as np
import pandas as pd

## 2. Cómo crear una Series

```python
s = pd.Series(data, index)
```

- `data`: los datos (lista, diccionario, array de NumPy...).
- `index`: las etiquetas emparejadas con los datos (si no se indica, pandas usa `0, 1, 2, ...`).

Veamos las formas más habituales:

In [2]:
# Desde una lista de datos, con índice explícito
primas = [420.0, 310.5, 275.0, 590.25, 180.0]
polizas = ["POL001", "POL002", "POL003", "POL004", "POL005"]

s = pd.Series(data=primas, index=polizas)
s

POL001    420.00
POL002    310.50
POL003    275.00
POL004    590.25
POL005    180.00
dtype: float64

In [3]:
# Desde un diccionario: las claves se convierten en índice automáticamente
primas_por_poliza = {"POL001": 420.0, "POL002": 310.5, "POL003": 275.0}
s_dict = pd.Series(primas_por_poliza)
s_dict

POL001    420.0
POL002    310.5
POL003    275.0
dtype: float64

In [4]:
# Desde un array de NumPy (sin índice explícito: usa 0, 1, 2...)
s_array = pd.Series(np.round(np.random.default_rng(seed=1).normal(400, 80, size=6), 2))
s_array

0    427.65
1    465.73
2    426.43
3    295.75
4    472.43
5    435.71
dtype: float64

In [5]:
# Se puede acceder a los datos como si fuera una lista (posición)...
print(s_array[:3])

# ...o como si fuera un diccionario (etiqueta de índice)
print(s["POL002"])

0    427.65
1    465.73
2    426.43
dtype: float64
310.5


> **Nota**: en la práctica, casi siempre crearás Series a partir de un archivo externo (un `.csv`, por ejemplo) o, más habitualmente, seleccionando una columna de un DataFrame — lo verás en el próximo notebook.

## 3. Series vs. listas de Python

Igual que un array de NumPy es más que una lista, una Series es más que un array: añade un índice con significado y un catálogo de métodos pensados para análisis de datos. Trabajemos con una Series de primas de una cartera pequeña de ejemplo.

In [6]:
primas_cartera = pd.Series(
    data=[420.0, 310.5, 275.0, 590.25, 180.0, 512.3, 275.0, 640.0],
    index=[f"POL{i:03d}" for i in range(1, 9)],
    name="prima_anual",
)
primas_cartera

POL001    420.00
POL002    310.50
POL003    275.00
POL004    590.25
POL005    180.00
POL006    512.30
POL007    275.00
POL008    640.00
Name: prima_anual, dtype: float64

### Atributos

In [7]:
print("dtype (tipo de dato):", primas_cartera.dtype)
print("size (número de elementos):", primas_cartera.size)
print("values (los datos, como array de NumPy):", primas_cartera.values)
print("index (las etiquetas):", primas_cartera.index)

dtype (tipo de dato): float64
size (número de elementos): 8
values (los datos, como array de NumPy): [420.   310.5  275.   590.25 180.   512.3  275.   640.  ]
index (las etiquetas): Index(['POL001', 'POL002', 'POL003', 'POL004', 'POL005', 'POL006', 'POL007',
       'POL008'],
      dtype='object')


Fíjate en `primas_cartera.values`: es literalmente un array de NumPy — la conexión entre lo que acabas de ver y lo que ves ahora no es casualidad.

In [8]:
print(type(primas_cartera.values))

<class 'numpy.ndarray'>


### Métodos más comunes

`.head()` y `.tail()` muestran los primeros/últimos elementos (por defecto, 5):

In [9]:
print(primas_cartera.head(3))
print(primas_cartera.tail(2))

POL001    420.0
POL002    310.5
POL003    275.0
Name: prima_anual, dtype: float64
POL007    275.0
POL008    640.0
Name: prima_anual, dtype: float64


`.sort_values()` ordena por valor; `.sort_index()` ordena por índice. Ambos aceptan `ascending` (`True` por defecto) e `inplace` (`False` por defecto — si es `True`, modifica la Series original y no devuelve nada).

In [10]:
print(primas_cartera.sort_values().head(3))       # 3 primas más bajas
print(primas_cartera.sort_values(ascending=False).head(3))  # 3 primas más altas
print(primas_cartera.sort_index().head(3))

POL005    180.0
POL003    275.0
POL007    275.0
Name: prima_anual, dtype: float64
POL008    640.00
POL004    590.25
POL006    512.30
Name: prima_anual, dtype: float64
POL001    420.0
POL002    310.5
POL003    275.0
Name: prima_anual, dtype: float64


> ⚠️ **Error típico**: usar `inplace=True` y esperar además que el método devuelva algo. Con `inplace=True`, la Series se modifica en el sitio y el método devuelve `None` — si escribes `primas_cartera = primas_cartera.sort_values(inplace=True)`, `primas_cartera` termina valiendo `None`.

`.apply()` aplica una función a cada valor — el equivalente en pandas de "arrastrar una fórmula" en Excel.

In [11]:
def clasificar_prima(valor: float) -> str:
    if valor >= 500:
        return "alta"
    elif valor >= 300:
        return "media"
    return "baja"


primas_cartera.apply(clasificar_prima)

POL001    media
POL002    media
POL003     baja
POL004     alta
POL005     baja
POL006     alta
POL007     baja
POL008     alta
Name: prima_anual, dtype: object

In [12]:
# También con lambdas, para transformaciones simples
primas_cartera.apply(lambda p: round(p * 1.08, 2))  # prima con IPS

POL001    453.60
POL002    335.34
POL003    297.00
POL004    637.47
POL005    194.40
POL006    553.28
POL007    297.00
POL008    691.20
Name: prima_anual, dtype: float64

### Reducciones numéricas

Los mismos resúmenes estadísticos que ya conoces de tu trabajo actuarial, como método directo sobre la Series:

In [13]:
print("Suma:", primas_cartera.sum())
print("Media:", round(primas_cartera.mean(), 2))
print("Desviación estándar:", round(primas_cartera.std(), 2))
print("Mediana:", primas_cartera.median())
print("Mínimo / máximo:", primas_cartera.min(), "/", primas_cartera.max())
print()
print("Resumen completo (.describe()):")
print(primas_cartera.describe())

Suma: 3203.05
Media: 400.38
Desviación estándar: 166.68
Mediana: 365.25
Mínimo / máximo: 180.0 / 640.0

Resumen completo (.describe()):
count      8.000000
mean     400.381250
std      166.675664
min      180.000000
25%      275.000000
50%      365.250000
75%      531.787500
max      640.000000
Name: prima_anual, dtype: float64


`.idxmin()` / `.idxmax()` devuelven la **etiqueta** del valor mínimo/máximo, no el valor en sí — muy útil para saber *qué póliza* tiene la prima más alta, no solo cuál es esa prima.

In [14]:
poliza_prima_max = primas_cartera.idxmax()
print(f"La póliza con la prima más alta es {poliza_prima_max}, con {primas_cartera[poliza_prima_max]}")

La póliza con la prima más alta es POL008, con 640.0


`.value_counts()` cuenta cuántas veces aparece cada valor — muy útil sobre columnas categóricas, que verás en el próximo notebook, pero funciona igual sobre cualquier Series:

In [15]:
primas_cartera.value_counts()

prima_anual
275.00    2
420.00    1
310.50    1
590.25    1
180.00    1
512.30    1
640.00    1
Name: count, dtype: int64

## 4. Vectorización de operaciones

Igual que con NumPy, las operaciones aritméticas entre Series **se alinean por índice**, no por posición. Esto es distinto de lo que pasaría con dos listas de Python del mismo tamaño (que se combinarían posición a posición sin comprobar nada).

In [16]:
primas_2023 = pd.Series({"POL001": 400.0, "POL002": 300.0, "POL003": 270.0})
primas_2024 = pd.Series({"POL002": 310.5, "POL003": 275.0, "POL004": 590.25})

# Se alinean por índice (nombre de póliza), no por posición
primas_2023 + primas_2024

POL001      NaN
POL002    610.5
POL003    545.0
POL004      NaN
dtype: float64

`POL001` y `POL004` dan `NaN` porque solo existen en una de las dos Series — pandas no intenta "adivinar" un emparejamiento por posición, respeta el índice. Es el mismo principio de alineación mencionado al principio del notebook, y es una diferencia importante frente a sumar dos listas de Python (que fallaría o combinaría por posición sin ningún criterio de negocio).

> 📊 **En Excel esto sería...** como sumar dos rangos con `BUSCARV` por clave en vez de sumar celda a celda por posición: si una clave no está en ambas tablas, el resultado para esa fila queda en blanco/error, en vez de sumar valores de filas que no tienen nada que ver entre sí.

In [17]:
# Con un escalar, funciona como ya viste en NumPy (broadcasting)
print(primas_cartera + 20)
print(primas_cartera * 1.08)

POL001    440.00
POL002    330.50
POL003    295.00
POL004    610.25
POL005    200.00
POL006    532.30
POL007    295.00
POL008    660.00
Name: prima_anual, dtype: float64
POL001    453.600
POL002    335.340
POL003    297.000
POL004    637.470
POL005    194.400
POL006    553.284
POL007    297.000
POL008    691.200
Name: prima_anual, dtype: float64


**Para ti:** crea dos Series de siniestros por póliza para dos periodos distintos, con solo dos claves en común entre ambas, y súmalas. Antes de ejecutar, predice cuántos `NaN` va a haber en el resultado — luego comprueba si acertaste.

In [18]:
# Tu código aquí


## 5. Filtros

Los filtros booleanos que viste con NumPy funcionan exactamente igual sobre una Series — es el mismo mecanismo, ahora respetando también el índice.

In [19]:
print(primas_cartera > 400)          # máscara booleana
print(primas_cartera[primas_cartera > 400])  # filtrado real

POL001     True
POL002    False
POL003    False
POL004     True
POL005    False
POL006     True
POL007    False
POL008     True
Name: prima_anual, dtype: bool
POL001    420.00
POL004    590.25
POL006    512.30
POL008    640.00
Name: prima_anual, dtype: float64


Para combinar condiciones se usan `&` (y) y `|` (o), **con cada condición entre paréntesis** — no se pueden usar `and`/`or` de Python normales sobre una Series completa.

In [20]:
# Primas en la cola de la distribución: muy altas o muy bajas
media, desv = primas_cartera.mean(), primas_cartera.std()

primas_cartera[
    (primas_cartera > media + desv)
    | (primas_cartera < media - desv)
]

POL004    590.25
POL005    180.00
POL008    640.00
Name: prima_anual, dtype: float64

> ⚠️ **Error típico**: escribir `primas_cartera > 300 and primas_cartera < 500` en vez de `(primas_cartera > 300) & (primas_cartera < 500)`. El `and`/`or` de Python esperan un único `True`/`False`, no una Series entera de ellos — con una Series de más de un elemento, Python lanza `ValueError: The truth value of a Series is ambiguous`. Es un error que verás muchas veces al empezar; el mensaje ya te dice justo lo que falla.

In [21]:
try:
    primas_cartera[primas_cartera > 300 and primas_cartera < 500]
except ValueError as e:
    print("Error esperado:", e)

Error esperado: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


**Para ti:** filtra `primas_cartera` para quedarte solo con las pólizas cuya prima es un número par truncado (pista: repasa el módulo `%` de la Unidad 2, y combínalo con `.apply()` para truncar cada valor con `int()` antes de comprobar la paridad).

In [22]:
# Tu código aquí


## Resumen

- Una `Series` es un array de NumPy con **índice explícito** — el vínculo entre valor e índice se mantiene salvo que lo rompas tú mismo.
- Se crea con `pd.Series(data, index)`; `data` puede ser una lista, un diccionario (las claves se vuelven índice) o un array de NumPy.
- `.values` de una Series es literalmente un array de NumPy — la conexión entre ambos notebooks no es casualidad.
- `.head()`/`.tail()`, `.sort_values()`/`.sort_index()` y `.apply()` son los métodos que más usarás para explorar y transformar una Series.
- Las reducciones numéricas (`.mean()`, `.std()`, `.describe()`...) y `.value_counts()` son tu primera parada para entender los datos.
- Las operaciones aritméticas entre dos Series **se alinean por índice**, no por posición — puede generar `NaN` donde una etiqueta no existe en ambas.
- Los filtros booleanos combinan condiciones con `&`/`|` (nunca `and`/`or`) y cada condición entre paréntesis.

**Siguiente:** `03_teoria_dataframes_basico.ipynb`, donde una Series se convierte en una columna de una tabla completa: el DataFrame.